
# Tiny Tao: Population Neural Networks for Modular Arithmetic on OSC GPUs

This notebook implements the core **Tiny Tao** experiment for modular multiplication using a
population of many very small neural networks trained in parallel on a GPU.

The key idea is to make the **population dimension part of the tensor computation**, rather than
training thousands of tiny networks in a Python loop. This is what makes the experiment a good
fit for OSC GPU nodes.

The default experiment uses the nonzero multiplicative group \(\mathbb F_p^\times\), a shared
2-D embedding \(z(a)=x_a+i y_a\), complex multiplication as the compositional primitive, and a
per-network linear readout.

It supports:

- full-table or held-row training;
- thousands of independently initialized networks in parallel;
- exact full-domain solver counts;
- GPU utilization / memory diagnostics;
- faithful-character analysis;
- kernel-2 / quotient+radial candidate analysis;
- saved checkpoints and CSV summaries.

The notebook is self-contained and designed to run inside **OSC OnDemand Jupyter** with one GPU.



## OSC launch notes

In OSC OnDemand, start a **Jupyter** interactive session and request a GPU in the launch form.
OSC's current Jupyter interface automatically selects an appropriate node type from the requested
compute resources.

For this experiment, start with:

- **1 GPU**
- **4–8 CPU cores**
- **16–32 GB system RAM**
- **2 hours wall time** for experimentation
- one ordinary Jupyter session, not Jupyter+LLM

Cardinal H100 or Ascend A100 hardware is ideal, but the notebook only assumes CUDA-capable PyTorch.

The first code cell verifies that the notebook actually landed on a GPU compute node.


In [ ]:

import os, sys, time, math, json, platform, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Stop here and relaunch the OSC Jupyter session with a GPU request."
    )

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA runtime seen by PyTorch:", torch.version.cuda)
print("GPU capability:", torch.cuda.get_device_capability(0))
print("Allocated now (GB):", torch.cuda.memory_allocated(0)/1e9)
print("Reserved now (GB):", torch.cuda.memory_reserved(0)/1e9)

print("\n--- nvidia-smi ---")
subprocess.run(
    ["nvidia-smi",
     "--query-gpu=name,utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu",
     "--format=csv,noheader,nounits"],
    check=False
)


## Experiment configuration

In [ ]:

# ---- Core settings ----
P = 13                   # prime modulus
POP = 1024               # number of independently initialized tiny networks
STEPS = 8000
LR = 3e-3
SEED = 20260909

# Split:
# "full"     -> train all pairs
# "held_row" -> hold a = P-1 out of training, while it remains present as b
SPLIT = "held_row"
HELD_A = P - 1

# Check exactness every this many steps.
CHECK_EVERY = 250

# Save output here.
RUN_DIR = Path.cwd() / "tiny_tao_results" / f"p{P}_{SPLIT}_{int(time.time())}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

print("Run directory:", RUN_DIR)



## Dataset

We work on the nonzero residues \(1,\ldots,p-1\). For prime \(p\), these form the cyclic group
\(\mathbb F_p^\times\).

For the held-row experiment, \(a=p-1\) is omitted **only from the first operand slot during
training**. Because the embedding is shared between the two operand slots, that residue is still
trained whenever it appears as \(b\). All output classes remain visible in the training set.


In [ ]:

def make_table(p):
    elems = torch.arange(1, p, dtype=torch.long)
    a, b = torch.meshgrid(elems, elems, indexing="ij")
    y = (a * b) % p

    # Convert displayed residue labels 1..p-1 into class indices 0..p-2.
    # For nonzero products modulo prime p, y is never 0.
    y_class = y - 1
    return elems, a.reshape(-1), b.reshape(-1), y_class.reshape(-1)

elems, a_all_cpu, b_all_cpu, y_all_cpu = make_table(P)

if SPLIT == "full":
    train_mask_cpu = torch.ones_like(a_all_cpu, dtype=torch.bool)
elif SPLIT == "held_row":
    train_mask_cpu = a_all_cpu != HELD_A
else:
    raise ValueError(SPLIT)

test_mask_cpu = ~train_mask_cpu

# Convert residue values 1..p-1 to embedding indices 0..p-2.
a_idx = (a_all_cpu - 1).to(device)
b_idx = (b_all_cpu - 1).to(device)
y_all = y_all_cpu.to(device)
train_mask = train_mask_cpu.to(device)
test_mask = test_mask_cpu.to(device)

print("Group order:", P-1)
print("Full pairs:", len(y_all_cpu))
print("Train pairs:", int(train_mask_cpu.sum()))
print("Held-out pairs:", int(test_mask_cpu.sum()))

train_classes = set(y_all_cpu[train_mask_cpu].tolist())
all_classes = set(range(P-1))
print("All output classes present in train:", train_classes == all_classes)



## Model C, vectorized across the population

For each independent network \(r\), learn one shared complex embedding for every residue,

\[
z_r(a)=x_r(a)+i\,y_r(a).
\]

Composition is fixed complex multiplication:

\[
h_r(a,b)=z_r(a)z_r(b).
\]

A network-specific linear readout maps the 2-D product to the \(p-1\) classes.

Every network has independent parameters, but all networks are trained simultaneously in one
set of GPU tensor operations.


In [ ]:

N = P - 1

# E[r, a, xy]
E = torch.empty(POP, N, 2, device=device, requires_grad=True)

# W[r, xy, class]
W = torch.empty(POP, 2, N, device=device, requires_grad=True)

# b[r, class]
bias = torch.zeros(POP, N, device=device, requires_grad=True)

with torch.no_grad():
    # Xavier-like scales
    E.normal_(mean=0.0, std=math.sqrt(2.0 / (N + 2)))
    W.normal_(mean=0.0, std=math.sqrt(2.0 / (2 + N)))

optimizer = torch.optim.Adam([E, W, bias], lr=LR)

param_per_net = N*2 + 2*N + N
print(f"Parameters/network: {param_per_net}")
print(f"Population parameters: {POP * param_per_net:,}")


In [ ]:

def forward_all(E, W, bias):
    # embeddings for every ordered pair
    ea = E[:, a_idx, :]   # [POP, pairs, 2]
    eb = E[:, b_idx, :]   # [POP, pairs, 2]

    xa, ya = ea[..., 0], ea[..., 1]
    xb, yb = eb[..., 0], eb[..., 1]

    # complex multiplication
    h = torch.stack(
        [xa*xb - ya*yb,
         xa*yb + ya*xb],
        dim=-1
    )                    # [POP, pairs, 2]

    logits = torch.einsum("rpi,ric->rpc", h, W) + bias[:, None, :]
    return logits, h

@torch.no_grad()
def accuracies(E, W, bias):
    logits, _ = forward_all(E, W, bias)
    pred = logits.argmax(dim=-1)
    target = y_all[None, :]

    correct = pred.eq(target)

    train_acc = correct[:, train_mask].float().mean(dim=1)
    test_acc = (
        correct[:, test_mask].float().mean(dim=1)
        if int(test_mask.sum()) > 0
        else torch.ones(POP, device=device)
    )
    domain_acc = correct.float().mean(dim=1)
    exact = correct.all(dim=1)

    return train_acc, test_acc, domain_acc, exact


## GPU throughput benchmark

In [ ]:

# Short warm-up + benchmark of the exact workload.
BENCH_STEPS = 100

for _ in range(10):
    optimizer.zero_grad(set_to_none=True)
    logits, _ = forward_all(E, W, bias)
    logits_train = logits[:, train_mask, :]
    targets = y_all[train_mask][None, :].expand(POP, -1)

    loss = F.cross_entropy(
        logits_train.reshape(-1, N),
        targets.reshape(-1),
        reduction="mean"
    )
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
t0 = time.perf_counter()

for _ in range(BENCH_STEPS):
    optimizer.zero_grad(set_to_none=True)
    logits, _ = forward_all(E, W, bias)
    logits_train = logits[:, train_mask, :]
    targets = y_all[train_mask][None, :].expand(POP, -1)

    loss = F.cross_entropy(
        logits_train.reshape(-1, N),
        targets.reshape(-1),
        reduction="mean"
    )
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

print(f"{BENCH_STEPS} steps: {elapsed:.3f} s")
print(f"steps/s: {BENCH_STEPS/elapsed:.2f}")
print(f"network-steps/s: {POP*BENCH_STEPS/elapsed:,.0f}")
print(f"estimated {STEPS} steps: {STEPS/(BENCH_STEPS/elapsed)/60:.1f} min")

subprocess.run(
    ["nvidia-smi",
     "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw,temperature.gpu",
     "--format=csv,noheader"],
    check=False
)



## Train the population

The loss is averaged across every network and every visible pair, but because each network has
its own parameters there is no information sharing between population members.

Exactness is evaluated using integer class predictions over the complete domain.


In [ ]:

history = []
torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()

for step in range(1, STEPS + 1):
    optimizer.zero_grad(set_to_none=True)

    logits, _ = forward_all(E, W, bias)
    logits_train = logits[:, train_mask, :]
    targets = y_all[train_mask][None, :].expand(POP, -1)

    loss = F.cross_entropy(
        logits_train.reshape(-1, N),
        targets.reshape(-1),
        reduction="mean"
    )

    loss.backward()
    optimizer.step()

    if step == 1 or step % CHECK_EVERY == 0 or step == STEPS:
        train_acc, test_acc, domain_acc, exact = accuracies(E, W, bias)
        exact_n = int(exact.sum())

        rec = {
            "step": step,
            "loss": float(loss.detach()),
            "mean_train_acc": float(train_acc.mean()),
            "mean_test_acc": float(test_acc.mean()),
            "mean_domain_acc": float(domain_acc.mean()),
            "exact": exact_n,
        }
        history.append(rec)

        print(
            f"{step:5d}  loss={rec['loss']:.6f}  "
            f"train={rec['mean_train_acc']:.4f}  "
            f"test={rec['mean_test_acc']:.4f}  "
            f"domain={rec['mean_domain_acc']:.4f}  "
            f"exact={exact_n}/{POP}"
        )

torch.cuda.synchronize()
train_seconds = time.perf_counter() - t0
print(f"\nTraining time: {train_seconds/60:.2f} min")
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")


## Save metrics and successful networks

In [ ]:

train_acc, test_acc, domain_acc, exact = accuracies(E, W, bias)
exact_ids = torch.where(exact)[0]

pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)

summary = {
    "p": P,
    "population": POP,
    "steps": STEPS,
    "lr": LR,
    "split": SPLIT,
    "held_a": HELD_A if SPLIT == "held_row" else None,
    "seed": SEED,
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "train_seconds": train_seconds,
    "exact_count": int(exact.sum()),
    "mean_train_accuracy": float(train_acc.mean()),
    "mean_test_accuracy": float(test_acc.mean()),
    "mean_domain_accuracy": float(domain_acc.mean()),
    "peak_gpu_memory_gb": torch.cuda.max_memory_allocated()/1e9,
}
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

if len(exact_ids):
    torch.save(
        {
            "network_ids": exact_ids.cpu(),
            "E": E.detach()[exact_ids].cpu(),
            "W": W.detach()[exact_ids].cpu(),
            "bias": bias.detach()[exact_ids].cpu(),
            "p": P,
            "split": SPLIT,
        },
        RUN_DIR / "successful.pt",
    )

summary



# Mechanistic analysis

The following cells look for the two solver families seen in the Tiny Tao experiments.

For prime \(p\), choose a primitive root \(g\) and assign each residue its discrete-log exponent
\(j\), so \(a=g^j\bmod p\).

A **faithful** one-circle solution has phase approximately

\[
\theta(a)=\phi+\frac{2\pi k j}{p-1},
\qquad \gcd(k,p-1)=1.
\]

A possible **kernel-2 / quotient+radial** solution has \(\gcd(k,p-1)=2\), with phase collapsing
\(a\) and \(-a\) and radius potentially carrying the missing binary coordinate.


In [ ]:

def prime_factors(n):
    out = set()
    d = 2
    while d*d <= n:
        while n % d == 0:
            out.add(d)
            n //= d
        d += 1
    if n > 1:
        out.add(n)
    return out

def primitive_root_prime(p):
    phi = p - 1
    fac = prime_factors(phi)
    for g in range(2, p):
        if all(pow(g, phi//q, p) != 1 for q in fac):
            return g
    raise RuntimeError("No primitive root found")

def discrete_log_table(p, g):
    table = {}
    x = 1
    for j in range(p-1):
        table[x] = j
        x = (x*g) % p
    return table

g = primitive_root_prime(P)
dlog = discrete_log_table(P, g)
j = torch.tensor([dlog[a] for a in range(1, P)], dtype=torch.float64)

print("Primitive root:", g)
print("Discrete-log order:", [pow(g, k, P) for k in range(P-1)])


In [ ]:

def wrap_angle(x):
    return torch.atan2(torch.sin(x), torch.cos(x))

def fit_windings(E_subset, p, j):
    # E_subset: [R, N, 2] on CPU
    z = E_subset[..., 0].double() + 1j * E_subset[..., 1].double()
    theta = torch.angle(z)
    radius = torch.abs(z)

    rows = []
    n = p - 1

    for r in range(E_subset.shape[0]):
        best = None
        for k in range(n):
            ideal = 2*math.pi*k*j/n
            # circular mean of theta - ideal gives global phase
            delta = theta[r] - ideal
            phi = torch.angle(torch.exp(1j*delta).mean())
            resid = wrap_angle(theta[r] - ideal - phi)
            mae_deg = float(resid.abs().mean() * 180/math.pi)

            candidate = (mae_deg, k, float(phi))
            if best is None or candidate < best:
                best = candidate

        mae_deg, k, phi = best
        rad = radius[r]
        rows.append({
            "net": r,
            "best_k": k,
            "gcd_k_n": math.gcd(k, n),
            "phase_mae_deg": mae_deg,
            "radius_cv": float(rad.std()/rad.mean()),
            "mean_radius": float(rad.mean()),
            "phi": phi,
        })

    return pd.DataFrame(rows)

if len(exact_ids):
    E_success = E.detach()[exact_ids].cpu()
    mech = fit_windings(E_success, P, j)
    display(mech.head(20))
    print("\nGCD counts:")
    print(mech["gcd_k_n"].value_counts().sort_index())
else:
    print("No exact networks to analyze.")



## Direct homomorphism diagnostic

After removing the global phase gauge, compare

\[
\theta(ab)
\]

with

\[
\theta(a)+\theta(b).
\]

For a true complex character, the centered residual should be close to zero over the entire table.


In [ ]:

@torch.no_grad()
def homomorphism_errors(E_subset, p):
    # CPU analysis
    z = E_subset[..., 0].double() + 1j*E_subset[..., 1].double()
    theta = torch.angle(z)

    aa = a_all_cpu - 1
    bb = b_all_cpu - 1
    cc = y_all_cpu  # class index = product residue - 1

    out = []
    for r in range(E_subset.shape[0]):
        delta = wrap_angle(theta[r, cc] - theta[r, aa] - theta[r, bb])
        gauge = torch.angle(torch.exp(1j*delta).mean())
        centered = wrap_angle(delta - gauge)
        out.append({
            "net": r,
            "mean_abs_deg": float(centered.abs().mean()*180/math.pi),
            "max_abs_deg": float(centered.abs().max()*180/math.pi),
            "gauge_deg": float(gauge*180/math.pi),
        })
    return pd.DataFrame(out)

if len(exact_ids):
    hom = homomorphism_errors(E_success, P)
    mech2 = mech.merge(hom, on="net")
    display(mech2.sort_values("mean_abs_deg").head(20))
    mech2.to_csv(RUN_DIR / "mechanistic_summary.csv", index=False)



## Kernel-2 radial diagnostic

For a kernel-2 winding, the angular representation collapses two elements in each fiber.
For primes \(p\equiv3\pmod4\), the p=11-style alternative solver may encode the missing
\(C_2\) coordinate in the radius.

This cell checks whether radius separates even and odd discrete-log parity.


In [ ]:

if len(exact_ids):
    zsucc = E_success[..., 0].double() + 1j*E_success[..., 1].double()
    rsucc = torch.abs(zsucc)

    parity = (j.long() % 2)
    even_mask = parity == 0
    odd_mask = parity == 1

    rows = []
    for r in range(len(exact_ids)):
        re = float(rsucc[r, even_mask].mean())
        ro = float(rsucc[r, odd_mask].mean())
        rows.append({
            "net": r,
            "rho_even": re,
            "rho_odd": ro,
            "q_odd_over_even": ro/re if re != 0 else np.nan,
            "radius_parity_separation":
                abs(re-ro) / ((re+ro)/2) if (re+ro) else np.nan,
        })

    radial = pd.DataFrame(rows)
    mech3 = mech2.merge(radial, on="net")
    display(
        mech3.sort_values(
            ["gcd_k_n", "phase_mae_deg"]
        ).head(30)
    )
    mech3.to_csv(RUN_DIR / "mechanistic_with_radius.csv", index=False)



# GPU utilization after the run

Within a live Jupyter session, `nvidia-smi` gives an instantaneous view. OSC also provides
post-job GPU accounting tools such as `gpu-seff` / `osc-seff` when you know the Slurm job ID.

The next cell prints the Slurm environment and current GPU state.


In [ ]:

print("SLURM_JOB_ID:", os.environ.get("SLURM_JOB_ID"))
print("SLURM_JOB_NAME:", os.environ.get("SLURM_JOB_NAME"))
print("SLURM_JOB_NODELIST:", os.environ.get("SLURM_JOB_NODELIST"))

subprocess.run(["nvidia-smi"], check=False)

print("\nIf gpu-seff is available, after the job/session completes you can run:")
if os.environ.get("SLURM_JOB_ID"):
    print(f"gpu-seff -v {os.environ['SLURM_JOB_ID']}")
    print(f"osc-seff {os.environ['SLURM_JOB_ID']}")
else:
    print("gpu-seff -v <jobid>")
    print("osc-seff <jobid>")



# Suggested OSC scaling experiments

Once the notebook works unchanged, useful tests are:

1. Run `POP = 1024, 4096, 16384` and compare network-steps/s.
2. Compare one OSC GPU against your local RTX 5060 Ti using the benchmark cell.
3. Keep float32 for the scientific experiment unless precision is explicitly being tested.
4. Record exact-solver frequency separately for every hardware/precision condition.
5. If using multiple GPUs later, split independent population members across GPUs rather than
   synchronizing individual tiny networks. The experiment is embarrassingly parallel at the
   population level.

A Cardinal H100 or Ascend A100 has far more VRAM than this experiment normally requires, so the
most useful OSC advantage may be the ability to run **much larger populations and independent
replicates simultaneously**, not merely making one 1024-network population faster.
